In [2]:
import pandas as pd
import requests
from datetime import datetime
import time

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from IPython.display import JSON

In [5]:
columns = [
    'id', 'brand', 'source', 'text', 'url', 
    'timestamp', 'sentiment', 'topic', 'urgency'
]

brand_mentions_df = pd.DataFrame(columns=columns)

In [6]:
brand_mentions_df

,id,brand,source,text,url,timestamp,sentiment,topic,urgency


In [7]:
response = requests.get("https://hn.algolia.com/api/v1/search_by_date?query=OpenAI&tags=comment&hitsPerPage=5").json()
response

{'exhaustive': {'nbHits': False, 'typo': False},
 'exhaustiveNbHits': False,
 'exhaustiveTypo': False,
 'hits': [{'_highlightResult': {'author': {'matchLevel': 'none',
     'matchedWords': [],
     'value': 'peterbell_nyc'},
    'comment_text': {'fullyHighlighted': False,
     'matchLevel': 'full',
     'matchedWords': ['openai'],
     'value': "Pi is a nice multi-agent wrapper. I use it to wrap my <em>OpenAI</em> max plan calls and my API calls. It takes care of some of the agent plumbing - still need sandbox, orchestrator, compounding, context, evals, etc but it's a nice component."},
    'story_title': {'matchLevel': 'none',
     'matchedWords': [],
     'value': 'Apache Burr: Build reliable AI agents and applications'},
    'story_url': {'matchLevel': 'none',
     'matchedWords': [],
     'value': 'https://burr.apache.org/'}},
   '_tags': ['comment', 'author_peterbell_nyc', 'story_48477400'],
   'author': 'peterbell_nyc',
   'comment_text': 'Pi is a nice multi-agent wrapper. I use 

In [8]:
response.get('hits')[1]

{'_highlightResult': {'author': {'matchLevel': 'none',
   'matchedWords': [],
   'value': 'peterbell_nyc'},
  'comment_text': {'fullyHighlighted': False,
   'matchLevel': 'full',
   'matchedWords': ['openai'],
   'value': "For me the heart of an agentic system is NOT using agents (except when you really have to). Components of a working system include:\n- Pipelines/recipes to describe multi-step flows (deterministic, agentic and HiTL steps), loops, conditionals, exit-on's for max loop iteractions, etc\n- The logistics to actually run the model and HiTL steps reliably across multiple agent worker pools\n- Management and delivery (and security/governance and permissioning) of thick skills with code to do as much as possible \n- Context management so the right agents have the right context for the right sessions at the right time\n- Project management - ability to store and access tickets, dependencies, track progress, restart stuck ticket claims, etc\n- Transcript saving, memory features

In [9]:
JSON(response)

<IPython.core.display.JSON object>

In [10]:
extracted_data = []
for hit in response.get('hits',[]):
    extracted_data.append({
        'id': hit.get('objectID'),
        'brand': 'OpenAI',
        'source': 'Hacker News',
        'text': hit.get('comment_text'),
        'url': f"https://news.ycombinator.com/item?id={hit.get('objectID')}",
        'timestamp': datetime.fromtimestamp(hit.get('created_at_i')),
        'sentiment': None,
        'topic': None,
        'urgency': None
    })

brand_mentions_df = pd.DataFrame(extracted_data)

In [11]:
brand_mentions_df

,id,brand,source,text,url,timestamp,sentiment,topic,urgency
0,48479710,OpenAI,Hacker News,Pi is a nice multi-agent wrapper. I use it to ...,https://news.ycombinator.com/item?id=48479710,2026-06-10 23:01:02,None,None,None
1,48479655,OpenAI,Hacker News,For me the heart of an agentic system is NOT u...,https://news.ycombinator.com/item?id=48479655,2026-06-10 22:58:10,None,None,None
2,48479568,OpenAI,Hacker News,I still to this day is baffled by all the idio...,https://news.ycombinator.com/item?id=48479568,2026-06-10 22:51:44,None,None,None
3,48479062,OpenAI,Hacker News,"OpenAI, Anthropic and Google serve models, but...",https://news.ycombinator.com/item?id=48479062,2026-06-10 22:17:59,None,None,None
4,48478966,OpenAI,Hacker News,"I guess the same rules also applied to OpenAI,...",https://news.ycombinator.com/item?id=48478966,2026-06-10 22:11:56,None,None,None


In [12]:
brand_mentions_df["text"][1]

'For me the heart of an agentic system is NOT using agents (except when you really have to). Components of a working system include:\n- Pipelines&#x2F;recipes to describe multi-step flows (deterministic, agentic and HiTL steps), loops, conditionals, exit-on&#x27;s for max loop iteractions, etc\n- The logistics to actually run the model and HiTL steps reliably across multiple agent worker pools\n- Management and delivery (and security&#x2F;governance and permissioning) of thick skills with code to do as much as possible \n- Context management so the right agents have the right context for the right sessions at the right time\n- Project management - ability to store and access tickets, dependencies, track progress, restart stuck ticket claims, etc\n- Transcript saving, memory features and dreaming&#x2F;compounding capabilities so the agents continue to learn from each session\n- o11y for understanding whats happening, tracking costs and usages, etc\n- Evals and auto-tuning of prompts so 

In [13]:
import html

brand_mentions_df["text"] = brand_mentions_df["text"].str.replace(r'<[^<>]*>', ' ', regex=True)
brand_mentions_df["text"] = brand_mentions_df["text"].apply(html.unescape)
brand_mentions_df["text"] = brand_mentions_df["text"].str.replace(r'\s+', ' ', regex=True).str.strip()

In [14]:
brand_mentions_df

,id,brand,source,text,url,timestamp,sentiment,topic,urgency
0,48479710,OpenAI,Hacker News,Pi is a nice multi-agent wrapper. I use it to ...,https://news.ycombinator.com/item?id=48479710,2026-06-10 23:01:02,None,None,None
1,48479655,OpenAI,Hacker News,For me the heart of an agentic system is NOT u...,https://news.ycombinator.com/item?id=48479655,2026-06-10 22:58:10,None,None,None
2,48479568,OpenAI,Hacker News,I still to this day is baffled by all the idio...,https://news.ycombinator.com/item?id=48479568,2026-06-10 22:51:44,None,None,None
3,48479062,OpenAI,Hacker News,"OpenAI, Anthropic and Google serve models, but...",https://news.ycombinator.com/item?id=48479062,2026-06-10 22:17:59,None,None,None
4,48478966,OpenAI,Hacker News,"I guess the same rules also applied to OpenAI,...",https://news.ycombinator.com/item?id=48478966,2026-06-10 22:11:56,None,None,None


In [15]:
brand_mentions_df["text"][0]

"Pi is a nice multi-agent wrapper. I use it to wrap my OpenAI max plan calls and my API calls. It takes care of some of the agent plumbing - still need sandbox, orchestrator, compounding, context, evals, etc but it's a nice component."

In [33]:
texts = "\n--\n".join(brand_mentions_df[brand_mentions_df['brand'] == 'OpenAI']['text'].tolist())
texts

'Pi is a nice multi-agent wrapper. I use it to wrap my OpenAI max plan calls and my API calls. It takes care of some of the agent plumbing - still need sandbox, orchestrator, compounding, context, evals, etc but it\'s a nice component.\n--\nFor me the heart of an agentic system is NOT using agents (except when you really have to). Components of a working system include: - Pipelines/recipes to describe multi-step flows (deterministic, agentic and HiTL steps), loops, conditionals, exit-on\'s for max loop iteractions, etc - The logistics to actually run the model and HiTL steps reliably across multiple agent worker pools - Management and delivery (and security/governance and permissioning) of thick skills with code to do as much as possible - Context management so the right agents have the right context for the right sessions at the right time - Project management - ability to store and access tickets, dependencies, track progress, restart stuck ticket claims, etc - Transcript saving, mem

In [31]:
brand_mentions_df[brand_mentions_df['brand'] == 'OpenAI']['text'].tolist()

["Pi is a nice multi-agent wrapper. I use it to wrap my OpenAI max plan calls and my API calls. It takes care of some of the agent plumbing - still need sandbox, orchestrator, compounding, context, evals, etc but it's a nice component.",
 "For me the heart of an agentic system is NOT using agents (except when you really have to). Components of a working system include: - Pipelines/recipes to describe multi-step flows (deterministic, agentic and HiTL steps), loops, conditionals, exit-on's for max loop iteractions, etc - The logistics to actually run the model and HiTL steps reliably across multiple agent worker pools - Management and delivery (and security/governance and permissioning) of thick skills with code to do as much as possible - Context management so the right agents have the right context for the right sessions at the right time - Project management - ability to store and access tickets, dependencies, track progress, restart stuck ticket claims, etc - Transcript saving, memor